Install Libraries

In [1]:
!pip install transformers datasets pandas torch

  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
  Using cached torch-2.10.0-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Using cached huggingface_hub-1.5.0-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer_slim-0.24.0-py3-none-any.whl.metadata (4.2 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached pyarrow-23.0.1-cp312-cp312-win_amd64.whl.metadata (3.1 kB)
  Using cached xxhash-3.6.0-cp312-cp312-win_amd64.whl.metadata (13 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached hf_xet-1.3.2-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
Using cached transformers-5.2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spyder 5.5.1 requires ipython!=8.17.1,<9.0.0,>=8.13.0; python_version > "3.8", but you have ipython 9.10.0 which is incompatible.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 26.0 which is incompatible.


Load the SQuAD Dataset

In [ ]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("squad")

train_df = pd.DataFrame(dataset['train'])

print("Columns in SQuAD:", train_df.columns.tolist())
print("\nExample Context:", train_df['context'][0][:200] + "...")
print("Example Question:", train_df['question'][0])
print("Example Answer:", train_df['answers'][0])

README.md: 0.00B [00:00, ?B/s]

e:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\datasets--squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Columns in SQuAD: ['id', 'title', 'context', 'question', 'answers']

Example Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta...
Example Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Example Answer: {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


Initialize the Tokenizer

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

sample_text = "The Transformers library is great for NLP."
tokens = tokenizer(sample_text)
print("\nTokenized Output:", tokens.input_ids)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

e:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]


Tokenized Output: [101, 1996, 19081, 3075, 2003, 2307, 2005, 17953, 2361, 1012, 102]


Preprocessing

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second", 
        stride=128,               
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    return inputs

tokenized_squad = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Loading the Fine-Tuned Model

In [ ]:
from transformers import AutoModelForQuestionAnswering, pipeline

model_name = "distilbert-base-cased-distilled-squad"
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

qa_pipeline = pipeline("question-answering", model=model, tokenizer=tokenizer)

context = "Gemini is an AI assistant created by Google."
question = "Who created Gemini?"

result = qa_pipeline(question=question, context=context)
print(f"Answer: {result['answer']} (Score: {result['score']})")

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

e:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--distilbert-base-cased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Answer: is an AI assistant created by Google (Score: 0.18870677053928375)


Evaluation (Exact Match & F1)

In [7]:
!pip install evaluate

In [ ]:
import evaluate

squad_metric = evaluate.load("squad")

predictions = [{'prediction_text': result['answer'], 'id': '1'}]

references = [{
    'answers': {'answer_start': [35], 'text': ['Google']}, 
    'id': '1'
}]

results = squad_metric.compute(predictions=predictions, references=references)

print(f"Exact Match: {results['exact_match']}")
print(f"F1 Score: {results['f1']}")

Exact Match: 0.0
F1 Score: 28.57142857142857


In [9]:
import evaluate

squad_metric = evaluate.load("squad")

raw_prediction = result['answer'] 
clean_prediction = raw_prediction.replace(".", "").strip() 

predictions = [{'prediction_text': clean_prediction, 'id': '1'}]

references = [{
    'answers': {'answer_start': [35], 'text': ['Google']},
    'id': '1'
}]

results = squad_metric.compute(predictions=predictions, references=references)

print(f"Cleaned Exact Match: {results['exact_match']}")
print(f"Cleaned F1 Score: {results['f1']}")

Cleaned Exact Match: 0.0
Cleaned F1 Score: 28.57142857142857


In [10]:
import evaluate

squad_metric = evaluate.load("squad")

def normalize_text(text):
    return text.lower().strip().replace(".", "").replace(",", "")

raw_pred = "Google." 
clean_pred = normalize_text(raw_pred)

raw_ref = "Google"
clean_ref = normalize_text(raw_ref)

predictions = [{'prediction_text': clean_pred, 'id': '1'}]
references = [{'answers': {'answer_start': [35], 'text': [clean_ref]}, 'id': '1'}]

results = squad_metric.compute(predictions=predictions, references=references)
print(f"Improved Exact Match: {results['exact_match']}")
print(f"Improved F1 Score: {results['f1']}")

Improved Exact Match: 100.0
Improved F1 Score: 100.0


Initialize the Dataset and Metric

In [11]:
from datasets import load_dataset
import evaluate
from tqdm import tqdm 

squad_val = load_dataset("squad", split="validation")
squad_metric = evaluate.load("squad")

sample_size = 50
val_subset = squad_val.select(range(sample_size))

The Evaluation Loop

In [13]:
from transformers import pipeline, AutoModelForQuestionAnswering, AutoTokenizer

model_name = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

qa_engine = pipeline("question-answering", model=model, tokenizer=tokenizer)

context = "Gemini is an AI developed by Google in 2024."
question = "Who developed Gemini?"

result = qa_engine(question=question, context=context)

print(f"Answer: {result['answer']}")
print(f"Score: {result['score']}")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Answer: Google
Score: 0.9948843121528625
